# Download and Process Open-Meteo Weather Data

Run this notebook from the `EDA/` directory after cloning the repository. It rebuilds the large observed-weather file used by `EDA_open_meteo_data.ipynb`:

`Open_Meteo_combined_2019-05-01_to_2026-05-01.csv`

`Open_Meteo_regional_features_2019-05-01_to_2026-05-01.csv`

`Open_Meteo_previous_runs_day1_2024-03-01_to_2026-05-02`

`Open_Meteo_previous_runs_day1_regional_features_2024-03-01_to_2026-05-02`

The combined file and cache are written under `../raw_data/` and are intentionally not committed to GitHub because they are large. The cache is restartable: if the API or notebook stops midway, rerun the download cell and completed point-year CSVs will be skipped.

The data files can also be downloaded from the following url: (to be updated)

### Sampling Places

![German renewable-weather sampling points](german_sampling_points_map.png)

The sampling design is based on real German renewable generation geography:

- Wind generation is concentrated in northern Germany and offshore zones.
- Solar PV is strongest in Bavaria, Baden-Wuerttemberg, North Rhine-Westphalia,
  and other high-installation states.
- We only sample for wind and solar generation.
- Curtailment can be driven by different spatial patterns: northern wind
  congestion, offshore wind, or southern/distribution-grid solar pressure.

We then group these sampling points by their geographical position and main type of generation.

### Weather data from 2019-05-01 to 2026-05-01

In [ ]:
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen
import json
import time

import pandas as pd

API_BASE_URL = "https://archive-api.open-meteo.com/v1/archive"
START_DATE = "2019-05-01"
END_DATE = "2026-05-01"

RAW_DATA_PATH = Path("../raw_data")
CACHE_DIR = RAW_DATA_PATH / "open_meteo_cache"
COMBINED_PATH = RAW_DATA_PATH / f"Open_Meteo_combined_{START_DATE}_to_{END_DATE}.csv"
METADATA_PATH = RAW_DATA_PATH / "Open_Meteo_sampling_point_metadata.csv"
TEMP_COMBINED_PATH = RAW_DATA_PATH / f"Open_Meteo_combined_{START_DATE}_to_{END_DATE}.tmp.csv"

OVERWRITE_CACHE = False
REQUEST_SLEEP_SECONDS = 1.0
MAX_RETRIES = 5

HOURLY_VARIABLES = [
    "wind_speed_10m",
    "wind_speed_100m",
    "wind_direction_10m",
    "wind_direction_100m",
    "wind_gusts_10m",
    "surface_pressure",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "cloud_cover",
    "temperature_2m",
    "sunshine_duration",
    "precipitation",
    "snowfall",
    "relative_humidity_2m",
]


@dataclass(frozen=True)
class WeatherPoint:
    point_id: str
    region: str
    federal_state_or_zone: str
    generation_mode: str
    sampling_point: str
    latitude: float
    longitude: float
    feature_groups: tuple[str, ...]


SAMPLING_POINTS = [
    WeatherPoint("offshore_north_sea", "North Sea offshore", "German Bight", "offshore_wind", "Offshore North Sea proxy", 54.50, 7.50, ("offshore_wind_points",)),
    WeatherPoint("offshore_baltic_sea", "Baltic Sea offshore", "Baltic Sea", "offshore_wind", "Offshore Baltic proxy", 54.60, 13.80, ("offshore_wind_points",)),
    WeatherPoint("husum", "Schleswig-Holstein", "Schleswig-Holstein", "onshore_wind", "Husum", 54.48, 9.05, ("north_wind_points",)),
    WeatherPoint("kiel", "Schleswig-Holstein", "Schleswig-Holstein", "onshore_wind", "Kiel", 54.32, 10.13, ("north_wind_points",)),
    WeatherPoint("cuxhaven", "Lower Saxony coast", "Lower Saxony", "onshore_wind_offshore_connection", "Cuxhaven", 53.86, 8.69, ("north_wind_points",)),
    WeatherPoint("emden", "Lower Saxony west", "Lower Saxony", "onshore_wind", "Emden", 53.37, 7.21, ("north_wind_points",)),
    WeatherPoint("rostock", "Mecklenburg-Vorpommern", "Mecklenburg-Vorpommern", "onshore_wind", "Rostock", 54.09, 12.14, ("northeast_wind_points",)),
    WeatherPoint("cottbus", "Brandenburg north/east", "Brandenburg", "onshore_wind_utility_solar", "Cottbus", 51.76, 14.33, ("northeast_wind_points", "east_solar_points")),
    WeatherPoint("potsdam", "Brandenburg central", "Brandenburg", "onshore_wind_solar", "Potsdam", 52.39, 13.06, ("northeast_wind_points", "east_solar_points")),
    WeatherPoint("magdeburg", "Saxony-Anhalt", "Saxony-Anhalt", "onshore_wind_solar", "Magdeburg", 52.12, 11.63, ("northeast_wind_points", "east_solar_points")),
    WeatherPoint("dortmund", "North Rhine-Westphalia", "North Rhine-Westphalia", "solar_onshore_wind_load", "Dortmund", 51.51, 7.46, ("west_solar_load_points",)),
    WeatherPoint("cologne", "North Rhine-Westphalia", "North Rhine-Westphalia", "solar_load", "Cologne", 50.94, 6.96, ("west_solar_load_points",)),
    WeatherPoint("nuremberg", "Bavaria north", "Bavaria", "solar_pv", "Nuremberg", 49.45, 11.08, ("south_solar_points",)),
    WeatherPoint("munich", "Bavaria south", "Bavaria", "solar_pv", "Munich", 48.14, 11.58, ("south_solar_points",)),
    WeatherPoint("regensburg", "Bavaria east", "Bavaria", "solar_pv", "Regensburg", 49.01, 12.10, ("south_solar_points",)),
    WeatherPoint("stuttgart", "Baden-Wuerttemberg north", "Baden-Wuerttemberg", "solar_pv", "Stuttgart", 48.78, 9.18, ("south_solar_points",)),
    WeatherPoint("freiburg", "Baden-Wuerttemberg south", "Baden-Wuerttemberg", "solar_pv", "Freiburg", 47.99, 7.84, ("south_solar_points",)),
    WeatherPoint("leipzig", "Saxony", "Saxony", "solar_pv_eastern_grid", "Leipzig", 51.34, 12.37, ("east_solar_points",)),
]

print(f"Sampling points: {len(SAMPLING_POINTS)}")
print(f"Hourly variables: {len(HOURLY_VARIABLES)}")
print(f"Raw data directory: {RAW_DATA_PATH}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Combined output: {COMBINED_PATH}")


In [ ]:
METADATA_COLUMNS = [
    "point_id",
    "sampling_point",
    "federal_state_or_zone",
    "generation_mode",
    "feature_groups",
]
DROP_FROM_WEATHER = [
    "sampling_point",
    "federal_state_or_zone",
    "generation_mode",
]


def read_json_url_with_retries(url: str, max_retries: int = MAX_RETRIES, initial_sleep_seconds: float = 5.0) -> dict:
    sleep_seconds = initial_sleep_seconds
    for attempt in range(1, max_retries + 1):
        try:
            with urlopen(url, timeout=120) as response:
                return json.loads(response.read().decode("utf-8"))
        except HTTPError as error:
            retryable = error.code == 429 or 500 <= error.code < 600
            if not retryable or attempt == max_retries:
                raise
            print(f"HTTP {error.code}; retrying in {sleep_seconds:.0f}s ({attempt}/{max_retries})")
            time.sleep(sleep_seconds)
            sleep_seconds *= 2
        except (URLError, TimeoutError):
            if attempt == max_retries:
                raise
            print(f"Network error; retrying in {sleep_seconds:.0f}s ({attempt}/{max_retries})")
            time.sleep(sleep_seconds)
            sleep_seconds *= 2
    raise RuntimeError("Failed to read JSON after retries.")


def fetch_open_meteo_hourly_weather(point: WeatherPoint, start_date: str, end_date: str, variables: list[str]) -> pd.DataFrame:
    query = urlencode(
        {
            "latitude": point.latitude,
            "longitude": point.longitude,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": ",".join(variables),
            "timezone": "UTC",
            "wind_speed_unit": "ms",
        }
    )
    payload = read_json_url_with_retries(f"{API_BASE_URL}?{query}")
    weather = pd.DataFrame(payload["hourly"])
    weather.insert(0, "timestamp_utc", pd.to_datetime(weather.pop("time"), utc=True))
    weather.insert(1, "point_id", point.point_id)
    weather.insert(2, "sampling_point", point.sampling_point)
    weather.insert(3, "region", point.region)
    weather.insert(4, "federal_state_or_zone", point.federal_state_or_zone)
    weather.insert(5, "generation_mode", point.generation_mode)
    weather.insert(6, "feature_groups", "|".join(point.feature_groups))
    weather.insert(7, "latitude", point.latitude)
    weather.insert(8, "longitude", point.longitude)
    return weather


def split_date_range_by_year(start_date: str, end_date: str) -> list[tuple[str, str]]:
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)
    if start > end:
        raise ValueError("start_date must not be later than end_date.")
    return [
        (
            max(start, date(year, 1, 1)).isoformat(),
            min(end, date(year, 12, 31)).isoformat(),
        )
        for year in range(start.year, end.year + 1)
    ]


def build_cache_path(point: WeatherPoint, segment_start: str, segment_end: str, cache_dir: Path) -> Path:
    year = date.fromisoformat(segment_start).year
    full_year_path = cache_dir / f"{point.point_id}_{year}.csv"
    if full_year_path.exists():
        return full_year_path
    return cache_dir / f"{point.point_id}_{segment_start}_to_{segment_end}.csv"


def cache_point_date_range_weather(point: WeatherPoint, start_date: str, end_date: str, cache_dir: Path, variables: list[str], overwrite: bool = False) -> tuple[Path, bool]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = build_cache_path(point, start_date, end_date, cache_dir)
    if cache_path.exists() and not overwrite:
        return cache_path, False
    weather = fetch_open_meteo_hourly_weather(point, start_date, end_date, variables)
    weather.to_csv(cache_path, index=False)
    return cache_path, True


def collect_weather_cache(points: list[WeatherPoint], start_date: str, end_date: str, cache_dir: Path, variables: list[str], overwrite: bool = False, request_sleep_seconds: float = REQUEST_SLEEP_SECONDS) -> list[Path]:
    cache_paths = []
    for point in points:
        for segment_start, segment_end in split_date_range_by_year(start_date, end_date):
            cache_path, downloaded = cache_point_date_range_weather(
                point=point,
                start_date=segment_start,
                end_date=segment_end,
                cache_dir=cache_dir,
                variables=variables,
                overwrite=overwrite,
            )
            cache_paths.append(cache_path)
            print(f"{'Downloaded' if downloaded else 'Skipped existing'} {cache_path}")
            if downloaded:
                time.sleep(request_sleep_seconds)
    return cache_paths


def expected_cache_paths(points: list[WeatherPoint], start_date: str, end_date: str, cache_dir: Path) -> list[Path]:
    return [
        build_cache_path(point, segment_start, segment_end, cache_dir)
        for point in points
        for segment_start, segment_end in split_date_range_by_year(start_date, end_date)
    ]


def require_existing_cache_files(cache_paths: list[Path]) -> None:
    missing = [path for path in cache_paths if not path.exists()]
    if missing:
        missing_list = "\n".join(str(path) for path in missing)
        raise FileNotFoundError(f"Missing cache files:\n{missing_list}")


def merge_cached_weather_files(cache_paths: list[Path], output_path: Path, start_date: str | None = None, end_date: str | None = None) -> pd.DataFrame:
    frames = [pd.read_csv(path, parse_dates=["timestamp_utc"]) for path in cache_paths]
    merged = pd.concat(frames, ignore_index=True)
    if start_date is not None:
        merged = merged[merged["timestamp_utc"] >= pd.Timestamp(start_date, tz="UTC")]
    if end_date is not None:
        end_exclusive = pd.Timestamp(end_date, tz="UTC") + pd.Timedelta(days=1)
        merged = merged[merged["timestamp_utc"] < end_exclusive]
    merged = merged.sort_values(["timestamp_utc", "point_id"]).reset_index(drop=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(output_path, index=False)
    return merged


def read_csv_columns(csv_path: Path) -> list[str]:
    return pd.read_csv(csv_path, nrows=0).columns.tolist()


def write_point_metadata(weather_path: Path, metadata_path: Path) -> bool:
    weather_columns = set(read_csv_columns(weather_path))
    missing_columns = set(METADATA_COLUMNS) - weather_columns
    if missing_columns:
        expected_removed = set(DROP_FROM_WEATHER)
        if missing_columns == expected_removed and metadata_path.exists():
            metadata_columns = set(read_csv_columns(metadata_path))
            if set(METADATA_COLUMNS).issubset(metadata_columns):
                return False
        raise ValueError(f"Cannot create metadata; missing columns: {sorted(missing_columns)}")
    metadata = pd.read_csv(weather_path, usecols=METADATA_COLUMNS).drop_duplicates()
    if metadata["point_id"].duplicated().any():
        raise ValueError("A point_id has multiple conflicting metadata rows.")
    metadata.sort_values("point_id").to_csv(metadata_path, index=False)
    return True


def rewrite_weather_without_repeated_metadata(weather_path: Path, temp_weather_path: Path) -> bool:
    weather_columns = set(read_csv_columns(weather_path))
    present_drop_columns = set(DROP_FROM_WEATHER) & weather_columns
    if not present_drop_columns:
        return False
    if present_drop_columns != set(DROP_FROM_WEATHER):
        raise ValueError(f"Combined weather table is only partially simplified; found columns {sorted(present_drop_columns)}.")
    first_chunk = True
    for chunk in pd.read_csv(weather_path, chunksize=100_000):
        chunk = chunk.drop(columns=DROP_FROM_WEATHER)
        chunk.to_csv(temp_weather_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
        first_chunk = False
    temp_weather_path.replace(weather_path)
    return True



This step makes one request per sampling point per year-bounded date segment. Existing cache files are skipped unless `OVERWRITE_CACHE = True`.

In [ ]:
cache_paths = collect_weather_cache(
    points=SAMPLING_POINTS,
    start_date=START_DATE,
    end_date=END_DATE,
    cache_dir=CACHE_DIR,
    variables=HOURLY_VARIABLES,
    overwrite=OVERWRITE_CACHE,
    request_sleep_seconds=REQUEST_SLEEP_SECONDS,
)

print(f"Available cache files: {len(cache_paths)}")


This step creates a large file consumed by the EDA notebook under `../raw_data/`. If the combined file already exists, the merge is skipped.

In [ ]:
cache_paths = expected_cache_paths(
    points=SAMPLING_POINTS,
    start_date=START_DATE,
    end_date=END_DATE,
    cache_dir=CACHE_DIR,
)
require_existing_cache_files(cache_paths)

if COMBINED_PATH.exists():
    print(f"Skipped merge; combined file already exists: {COMBINED_PATH}")
else:
    combined = merge_cached_weather_files(
        cache_paths,
        COMBINED_PATH,
        start_date=START_DATE,
        end_date=END_DATE,
    )
    print(f"Wrote: {COMBINED_PATH}")
    print(f"Rows: {len(combined):,}")
    print(f"Points: {combined['point_id'].nunique()}")
    del combined


The combined weather table initially repeats stable point metadata on every hourly row. This cell writes a compact metadata file and removes repeated columns from the large weather CSV.

In [ ]:
metadata_written = write_point_metadata(COMBINED_PATH, METADATA_PATH)
weather_rewritten = rewrite_weather_without_repeated_metadata(COMBINED_PATH, TEMP_COMBINED_PATH)

print(f"{'Wrote' if metadata_written else 'Reused'} metadata: {METADATA_PATH}")
print(f"{'Updated' if weather_rewritten else 'Already simplified'} combined data: {COMBINED_PATH}")


Validate Outputs

In [ ]:
combined_sample = pd.read_csv(COMBINED_PATH, nrows=5)
metadata = pd.read_csv(METADATA_PATH)

print(f"Combined file: {COMBINED_PATH}")
print(f"Combined columns ({len(combined_sample.columns)}):")
print(combined_sample.columns.tolist())
print(f"Metadata file: {METADATA_PATH}")
print(f"Metadata rows: {len(metadata)}")
metadata.head()


## Build Observed Regional Weather Features

Read the observed combined weather CSV, select the core weather variables used for EDA/modeling, aggregate sampling points into regional wind/solar feature groups, add local-time columns, and write the processed regional feature table.

In [ ]:
import numpy as np

PROCESSED_DATA_PATH = Path("../processed_data")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
REGIONAL_FEATURES_PATH = PROCESSED_DATA_PATH / f"Open_Meteo_regional_features_{START_DATE}_to_{END_DATE}.csv"

offshore_point_regions = {"offshore_north_sea", "offshore_baltic_sea"}
regional_generation_type_by_region = {
    "offshore_north_sea": "wind",
    "offshore_baltic_sea": "wind",
    "north_wind": "wind",
    "northeast_wind": "wind",
    "south_solar": "solar",
    "west_solar_load": "solar",
    "east_solar": "solar",
}

regional_core_features = [
    "wind_speed_100m",
    "wind_direction_100m",
    "shortwave_radiation",
    "cloud_cover",
    "sunshine_duration",
    "temperature_2m",
]

observed_point_weather = pd.read_csv(
    COMBINED_PATH,
    usecols=["timestamp_utc", "point_id", "feature_groups", *regional_core_features],
    parse_dates=["timestamp_utc"],
)
observed_point_weather["region"] = observed_point_weather["feature_groups"].str.split("|")
observed_expanded_weather = observed_point_weather.explode("region", ignore_index=True)
observed_expanded_weather["region"] = observed_expanded_weather["region"].str.removesuffix("_points")
offshore_mask = observed_expanded_weather["point_id"].isin(offshore_point_regions)
observed_expanded_weather.loc[offshore_mask, "region"] = observed_expanded_weather.loc[offshore_mask, "point_id"]
observed_expanded_weather["generation_type"] = observed_expanded_weather["region"].map(regional_generation_type_by_region)

observed_direction_radians = np.deg2rad(observed_expanded_weather["wind_direction_100m"])
observed_expanded_weather["wind_direction_100m_sin"] = np.sin(observed_direction_radians)
observed_expanded_weather["wind_direction_100m_cos"] = np.cos(observed_direction_radians)

observed_regional_features = (
    observed_expanded_weather.groupby(["timestamp_utc", "region", "generation_type"], as_index=False)
    .agg(
        wind_speed_100m_avg=("wind_speed_100m", "mean"),
        wind_direction_100m_sin_avg=("wind_direction_100m_sin", "mean"),
        wind_direction_100m_cos_avg=("wind_direction_100m_cos", "mean"),
        solar_radiation_avg=("shortwave_radiation", "mean"),
        cloud_cover_avg=("cloud_cover", "mean"),
        sunshine_duration_avg=("sunshine_duration", "mean"),
        temperature_2m_avg=("temperature_2m", "mean"),
    )
)

observed_regional_features["timestamp_de"] = observed_regional_features["timestamp_utc"].dt.tz_convert("Europe/Berlin")
observed_regional_features["hour_local"] = observed_regional_features["timestamp_de"].dt.hour
observed_regional_features["day_of_week"] = observed_regional_features["timestamp_de"].dt.dayofweek
observed_regional_features["month"] = observed_regional_features["timestamp_de"].dt.month
observed_regional_features["is_weekend"] = observed_regional_features["day_of_week"] >= 5
observed_regional_features["season"] = observed_regional_features["month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
})

if observed_regional_features.duplicated(["timestamp_utc", "region"]).any():
    raise ValueError("Observed regional features contain duplicate timestamp-region keys.")

observed_regional_features.to_csv(REGIONAL_FEATURES_PATH, index=False)
print(f"Wrote: {REGIONAL_FEATURES_PATH}")
print(f"Rows: {len(observed_regional_features):,}")
print(f"Regions: {observed_regional_features['region'].nunique()}")
observed_regional_features.groupby(["region", "generation_type"]).size()


## Forecast Data

This section downloads Open-Meteo Previous Runs forecasts issued 24 hours before each valid timestamp. It uses the same sampling points and weather variables as the observed-weather download, but the available archive starts at `2024-03-01` for the variables used here.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

PREVIOUS_RUNS_API_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
PREVIOUS_DAY = 1
FORECAST_LEAD_HOURS = 24
PREVIOUS_RUNS_START_DATE = "2024-03-01"
PREVIOUS_RUNS_END_DATE = "2026-05-02"
PREVIOUS_RUNS_DATE_RANGE = f"{PREVIOUS_RUNS_START_DATE}_to_{PREVIOUS_RUNS_END_DATE}"

PREVIOUS_RUNS_CACHE_DIR = RAW_DATA_PATH / "previous_runs_day1_cache"
PREVIOUS_RUNS_COMBINED_PATH = RAW_DATA_PATH / f"Open_Meteo_previous_runs_day1_{PREVIOUS_RUNS_DATE_RANGE}.csv"
PREVIOUS_RUNS_TEMP_PATH = RAW_DATA_PATH / f"Open_Meteo_previous_runs_day1_{PREVIOUS_RUNS_DATE_RANGE}.tmp.csv"

FORECAST_WORKERS = 1

print(f"Forecast cache directory: {PREVIOUS_RUNS_CACHE_DIR}")
print(f"Forecast combined output: {PREVIOUS_RUNS_COMBINED_PATH}")
print(f"Expected forecast cache files: {len(SAMPLING_POINTS) * len(split_date_range_by_year(PREVIOUS_RUNS_START_DATE, PREVIOUS_RUNS_END_DATE))}")


In [ ]:
def fetch_open_meteo_previous_day1(point: WeatherPoint, start_date: str, end_date: str, variables: list[str]) -> pd.DataFrame:
    requested_variables = [f"{variable}_previous_day{PREVIOUS_DAY}" for variable in variables]
    query = urlencode(
        {
            "latitude": point.latitude,
            "longitude": point.longitude,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": ",".join(requested_variables),
            "timezone": "UTC",
            "wind_speed_unit": "ms",
        }
    )
    payload = read_json_url_with_retries(
        f"{PREVIOUS_RUNS_API_URL}?{query}",
        max_retries=MAX_RETRIES,
    )
    forecast = pd.DataFrame(payload["hourly"])
    forecast = forecast.rename(
        columns={f"{variable}_previous_day{PREVIOUS_DAY}": variable for variable in variables}
    )
    missing_columns = set(variables) - set(forecast.columns)
    if missing_columns:
        raise ValueError(f"Previous Runs response is missing variables: {sorted(missing_columns)}")
    empty_variables = [variable for variable in variables if forecast[variable].isna().all()]
    if empty_variables:
        raise ValueError(
            "Previous Runs day-1 data is unavailable for the requested segment; "
            f"all values are missing for {empty_variables}."
        )
    forecast.insert(0, "timestamp_utc", pd.to_datetime(forecast.pop("time"), utc=True))
    forecast.insert(1, "point_id", point.point_id)
    forecast.insert(2, "sampling_point", point.sampling_point)
    forecast.insert(3, "region", point.region)
    forecast.insert(4, "federal_state_or_zone", point.federal_state_or_zone)
    forecast.insert(5, "generation_mode", point.generation_mode)
    forecast.insert(6, "feature_groups", "|".join(point.feature_groups))
    forecast.insert(7, "latitude", point.latitude)
    forecast.insert(8, "longitude", point.longitude)
    forecast.insert(9, "forecast_lead_hours", FORECAST_LEAD_HOURS)
    return forecast


def build_forecast_cache_path(point: WeatherPoint, segment_start: str, segment_end: str, cache_dir: Path) -> Path:
    return cache_dir / f"{point.point_id}_{segment_start}_to_{segment_end}_previous_day1.csv"


def cache_point_previous_day1(point: WeatherPoint, start_date: str, end_date: str, cache_dir: Path, variables: list[str], overwrite: bool = False) -> tuple[Path, bool]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = build_forecast_cache_path(point, start_date, end_date, cache_dir)
    if cache_path.exists() and not overwrite:
        return cache_path, False
    forecast = fetch_open_meteo_previous_day1(
        point=point,
        start_date=start_date,
        end_date=end_date,
        variables=variables,
    )
    forecast.to_csv(cache_path, index=False)
    return cache_path, True


def collect_previous_day1_cache(points: list[WeatherPoint], start_date: str, end_date: str, cache_dir: Path, variables: list[str], overwrite: bool = False, request_sleep_seconds: float = REQUEST_SLEEP_SECONDS, workers: int = 1) -> list[Path]:
    if workers < 1:
        raise ValueError("workers must be at least 1.")
    tasks = [
        (point, segment_start, segment_end)
        for point in points
        for segment_start, segment_end in split_date_range_by_year(start_date, end_date)
    ]

    def collect_task(task: tuple[WeatherPoint, str, str]) -> Path:
        point, segment_start, segment_end = task
        cache_path, downloaded = cache_point_previous_day1(
            point=point,
            start_date=segment_start,
            end_date=segment_end,
            cache_dir=cache_dir,
            variables=variables,
            overwrite=overwrite,
        )
        print(f"{'Downloaded' if downloaded else 'Skipped existing'} {cache_path}")
        if downloaded:
            time.sleep(request_sleep_seconds)
        return cache_path

    if workers == 1:
        return [collect_task(task) for task in tasks]
    with ThreadPoolExecutor(max_workers=workers) as executor:
        return list(executor.map(collect_task, tasks))


def expected_previous_day1_cache_paths(points: list[WeatherPoint], start_date: str, end_date: str, cache_dir: Path) -> list[Path]:
    return [
        build_forecast_cache_path(point, segment_start, segment_end, cache_dir)
        for point in points
        for segment_start, segment_end in split_date_range_by_year(start_date, end_date)
    ]


def require_existing_previous_day1_cache_files(cache_paths: list[Path]) -> None:
    missing = [path for path in cache_paths if not path.exists()]
    if missing:
        missing_list = "\n".join(str(path) for path in missing)
        raise FileNotFoundError(f"Missing Previous Runs day-1 cache files:\n{missing_list}")


In [ ]:
previous_runs_cache_paths = collect_previous_day1_cache(
    points=SAMPLING_POINTS,
    start_date=PREVIOUS_RUNS_START_DATE,
    end_date=PREVIOUS_RUNS_END_DATE,
    cache_dir=PREVIOUS_RUNS_CACHE_DIR,
    variables=HOURLY_VARIABLES,
    overwrite=OVERWRITE_CACHE,
    request_sleep_seconds=REQUEST_SLEEP_SECONDS,
    workers=FORECAST_WORKERS,
)

print(f"Available forecast cache files: {len(previous_runs_cache_paths)}")


In [ ]:
previous_runs_cache_paths = expected_previous_day1_cache_paths(
    points=SAMPLING_POINTS,
    start_date=PREVIOUS_RUNS_START_DATE,
    end_date=PREVIOUS_RUNS_END_DATE,
    cache_dir=PREVIOUS_RUNS_CACHE_DIR,
)
require_existing_previous_day1_cache_files(previous_runs_cache_paths)

if PREVIOUS_RUNS_COMBINED_PATH.exists():
    print(f"Skipped forecast merge; combined file already exists: {PREVIOUS_RUNS_COMBINED_PATH}")
else:
    point_previous_runs = merge_cached_weather_files(
        previous_runs_cache_paths,
        PREVIOUS_RUNS_COMBINED_PATH,
        start_date=PREVIOUS_RUNS_START_DATE,
        end_date=PREVIOUS_RUNS_END_DATE,
    )
    print(f"Wrote: {PREVIOUS_RUNS_COMBINED_PATH}")
    print(f"Rows: {len(point_previous_runs):,}")
    print(f"Points: {point_previous_runs['point_id'].nunique()}")
    print(f"UTC range: {point_previous_runs['timestamp_utc'].min()} to {point_previous_runs['timestamp_utc'].max()}")
    del point_previous_runs


In [ ]:
previous_runs_metadata_written = write_point_metadata(PREVIOUS_RUNS_COMBINED_PATH, METADATA_PATH)
previous_runs_rewritten = rewrite_weather_without_repeated_metadata(
    PREVIOUS_RUNS_COMBINED_PATH,
    PREVIOUS_RUNS_TEMP_PATH,
)

forecast_sample = pd.read_csv(PREVIOUS_RUNS_COMBINED_PATH, nrows=5)
metadata = pd.read_csv(METADATA_PATH)

print(f"{'Wrote' if previous_runs_metadata_written else 'Reused'} metadata: {METADATA_PATH}")
print(f"{'Updated' if previous_runs_rewritten else 'Already simplified'} forecast data: {PREVIOUS_RUNS_COMBINED_PATH}")
print(f"Forecast columns ({len(forecast_sample.columns)}):")
print(forecast_sample.columns.tolist())
print(f"Metadata rows: {len(metadata)}")
forecast_sample.head()


## Build Forecast Regional Weather Features

Read the combined Previous Runs day-1 weather CSV and apply the same regional aggregation used for observed weather. This writes a forecast regional feature table with the same schema as the observed regional feature table.

In [ ]:
import numpy as np

PROCESSED_DATA_PATH = Path("../processed_data")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
PREVIOUS_RUNS_REGIONAL_PATH = PROCESSED_DATA_PATH / f"Open_Meteo_previous_runs_day1_regional_features_{PREVIOUS_RUNS_DATE_RANGE}.csv"

offshore_point_regions = {"offshore_north_sea", "offshore_baltic_sea"}
regional_generation_type_by_region = {
    "offshore_north_sea": "wind",
    "offshore_baltic_sea": "wind",
    "north_wind": "wind",
    "northeast_wind": "wind",
    "south_solar": "solar",
    "west_solar_load": "solar",
    "east_solar": "solar",
}

regional_core_features = [
    "wind_speed_100m",
    "wind_direction_100m",
    "shortwave_radiation",
    "cloud_cover",
    "sunshine_duration",
    "temperature_2m",
]

forecast_point_weather = pd.read_csv(
    PREVIOUS_RUNS_COMBINED_PATH,
    usecols=["timestamp_utc", "point_id", "feature_groups", *regional_core_features],
    parse_dates=["timestamp_utc"],
)
forecast_point_weather["region"] = forecast_point_weather["feature_groups"].str.split("|")
forecast_expanded_weather = forecast_point_weather.explode("region", ignore_index=True)
forecast_expanded_weather["region"] = forecast_expanded_weather["region"].str.removesuffix("_points")
offshore_mask = forecast_expanded_weather["point_id"].isin(offshore_point_regions)
forecast_expanded_weather.loc[offshore_mask, "region"] = forecast_expanded_weather.loc[offshore_mask, "point_id"]
forecast_expanded_weather["generation_type"] = forecast_expanded_weather["region"].map(regional_generation_type_by_region)

forecast_direction_radians = np.deg2rad(forecast_expanded_weather["wind_direction_100m"])
forecast_expanded_weather["wind_direction_100m_sin"] = np.sin(forecast_direction_radians)
forecast_expanded_weather["wind_direction_100m_cos"] = np.cos(forecast_direction_radians)

forecast_regional_features = (
    forecast_expanded_weather.groupby(["timestamp_utc", "region", "generation_type"], as_index=False)
    .agg(
        wind_speed_100m_avg=("wind_speed_100m", "mean"),
        wind_direction_100m_sin_avg=("wind_direction_100m_sin", "mean"),
        wind_direction_100m_cos_avg=("wind_direction_100m_cos", "mean"),
        solar_radiation_avg=("shortwave_radiation", "mean"),
        cloud_cover_avg=("cloud_cover", "mean"),
        sunshine_duration_avg=("sunshine_duration", "mean"),
        temperature_2m_avg=("temperature_2m", "mean"),
    )
)

forecast_regional_features["timestamp_de"] = forecast_regional_features["timestamp_utc"].dt.tz_convert("Europe/Berlin")
forecast_regional_features["hour_local"] = forecast_regional_features["timestamp_de"].dt.hour
forecast_regional_features["day_of_week"] = forecast_regional_features["timestamp_de"].dt.dayofweek
forecast_regional_features["month"] = forecast_regional_features["timestamp_de"].dt.month
forecast_regional_features["is_weekend"] = forecast_regional_features["day_of_week"] >= 5
forecast_regional_features["season"] = forecast_regional_features["month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
})

if forecast_regional_features.duplicated(["timestamp_utc", "region"]).any():
    raise ValueError("Forecast regional features contain duplicate timestamp-region keys.")

forecast_regional_features.to_csv(PREVIOUS_RUNS_REGIONAL_PATH, index=False)
print(f"Wrote: {PREVIOUS_RUNS_REGIONAL_PATH}")
print(f"Rows: {len(forecast_regional_features):,}")
print(f"Regions: {forecast_regional_features['region'].nunique()}")
forecast_regional_features.groupby(["region", "generation_type"]).size()
